In [87]:
from gas.gas_circuit import GAS_HW, bin_to_int, GAS_dicke
from gas.dicke import dicke_state

In [88]:
import numpy as np 
import json
import os 
import h5py
from qiskit_aer import StatevectorSimulator
from qiskit import transpile, QuantumCircuit

from symmer.operators.base import PauliwordOp, QuantumState

from sympy import var
import h5py
from itertools import combinations
import matplotlib.pyplot as plt

In [89]:
cwd = os.getcwd()
par_dir = os.path.dirname(os.path.dirname(cwd))
h5_data = os.path.join(par_dir, "notebooks", "H3_data.h5")
assert os.path.isfile(h5_data), "H3_data.h5 not found"

In [90]:
## load data
with h5py.File(h5_data, "r") as infile:

    f_op = PauliwordOp(infile["F_true_symp_matrix"][()],
                       infile["F_true_coeff_vec"][()])

    f_op_rounded = PauliwordOp(infile["F_rounded_symp_matrix"][()],
                                infile["F_rounded_coeff_vec"][()])
    
    linear_true = dict(zip(infile["lin_true_keys"][()],
                       infile["lin_true_values"][()]))
    
    quad_true = dict(zip( [(a,b) for a,b in infile["quad_true_keys"][()]],
                       infile["quad_true_values"][()]))
    

    linear_rounded = dict(zip(infile["lin_rounded_keys"][()],
                    infile["lin_rounded_values"][()]))
    
    quad_rounded   = dict(zip([(a,b) for a,b in infile["quad_rounded_keys"][()]],
                         infile["quad_rounded_values"][()]))
    
    n_elec = infile["SCF_nelec"][()]
    basis_set = infile["SCF_basis"][()]
    n_spin = 2*infile["h1e_MO"][()].shape[0]

print("LOADED")

LOADED


In [91]:
### write QUBO problem!
variables = [var(f'x{q_idx}', bool=True) for q_idx in range(n_spin)]

problem_fn = 0
for q_idx, coeff in linear_rounded.items():
    problem_fn+= variables[q_idx]* int(coeff)

for (q_idx1, q_idx2), coeff in quad_rounded.items():
    problem_fn+= variables[q_idx1]*variables[q_idx2]* int(coeff)

problem_fn

4*x0*x1 + 4*x0*x2 + 4*x0*x3 + 4*x0*x4 + 4*x0*x5 - 14*x0 + 4*x1*x2 + 4*x1*x3 + 4*x1*x4 + 4*x1*x5 - 14*x1 + 4*x2*x3 + 4*x2*x4 + 4*x2*x5 - 10*x2 + 4*x3*x4 + 4*x3*x5 - 10*x3 + 8*x4*x5 - 6*x4 - 6*x5

In [ ]:
### get equation in latex

from sympy.printing.latex import print_latex
print_latex(problem_fn)

4 x_{0} x_{1} + 4 x_{0} x_{2} + 4 x_{0} x_{3} + 4 x_{0} x_{4} + 4 x_{0} x_{5} - 14 x_{0} + 4 x_{1} x_{2} + 4 x_{1} x_{3} + 4 x_{1} x_{4} + 4 x_{1} x_{5} - 14 x_{1} + 4 x_{2} x_{3} + 4 x_{2} x_{4} + 4 x_{2} x_{5} - 10 x_{2} + 4 x_{3} x_{4} + 4 x_{3} x_{5} - 10 x_{3} + 8 x_{4} x_{5} - 6 x_{4} - 6 x_{5}


In [93]:
## go through all valid solutions (classical brute force)
def get_S2(state_mat):
    na = state_mat[0::2].sum()
    nb = state_mat[0::2].sum()
    ab = np.logical_and(state_mat[0::2], state_mat[1::2]).sum()
    a_minus_b_over_2 = (na-nb)/2
    S2_val = a_minus_b_over_2**2 + a_minus_b_over_2 + nb -ab
    return S2_val


def n_choose_k_vectors(n: int, k: int):
    return np.array(
        [
            [int(i in comb) for i in range(n)]
            for comb in combinations(np.arange(n), k)
        ]
    )

alph_bitstrings = n_choose_k_vectors(f_op.n_qubits//2, n_elec[0])
beta_bitstrings = n_choose_k_vectors(f_op.n_qubits//2, n_elec[1])

eigstate_and_vals = dict()
for alpha_term in alph_bitstrings:
    for beta_term in beta_bitstrings:
        state = np.vstack((alpha_term, beta_term)).T.reshape([f_op.n_qubits]) 
        evaluated = int(problem_fn.subs(zip(variables, state)))
        # print(int(evaluated))
        eigstate_and_vals[tuple(state.tolist())] = (evaluated, get_S2(state))
        # print(''.join(list(map(str, state.tolist()))))
eigstate_and_vals

{(1, 1, 1, 1, 0, 0): (-24, np.float64(0.0)),
 (1, 1, 1, 0, 0, 1): (-20, np.float64(1.0)),
 (1, 0, 1, 1, 0, 1): (-16, np.float64(1.0)),
 (1, 1, 0, 1, 1, 0): (-20, np.float64(1.0)),
 (1, 1, 0, 0, 1, 1): (-12, np.float64(0.0)),
 (1, 0, 0, 1, 1, 1): (-8, np.float64(1.0)),
 (0, 1, 1, 1, 1, 0): (-16, np.float64(1.0)),
 (0, 1, 1, 0, 1, 1): (-8, np.float64(1.0)),
 (0, 0, 1, 1, 1, 1): (-4, np.float64(0.0))}

In [189]:
from collections import Counter
Counter([val[0] for val in eigstate_and_vals.values()])

Counter({-20: 2, -16: 2, -8: 2, -24: 1, -12: 1, -4: 1})

In [94]:
nq_qubo = f_op.n_qubits
maxval_qubo = np.abs(list(linear_rounded.values())).sum() + np.abs(list(quad_rounded.values())).sum()

print(f"number of QUBO qubits: {nq_qubo}")
print(f"maximum QUBO value possible: {maxval_qubo}")

number of QUBO qubits: 6
maximum QUBO value possible: 124.0


In [95]:
nq_fx = int(np.ceil(np.log2(maxval_qubo))) + 1 # add one for negative values
print(f"maximum of f(x) qubits: {nq_fx}")

## get binary value for all 1s state (max value) for na and nb counts!
nq_na = int(np.ceil(np.log2(bin_to_int(np.array([1]* (nq_qubo//2)))))) 
nq_nb = int(np.ceil(np.log2(bin_to_int(np.array([1]* (nq_qubo//2)))))) 

print(f"maximum of Na(x) qubits: {nq_na}") # alpha electrons
print(f"maximum of Nb(x) qubits: {nq_nb}") # beta electrons

## get binary value for all 1s state (max value) for na and nb counts!
na_target = int(n_elec[0])
nb_target = int(n_elec[1])


maximum of f(x) qubits: 8
maximum of Na(x) qubits: 3
maximum of Nb(x) qubits: 3


In [96]:
total_nq = nq_qubo + nq_fx + nq_na + nq_nb + 3
print(f"total number of qubits in GAS-SCF (non-Dicke): {total_nq}")

total number of qubits in GAS-SCF (non-Dicke): 23


# Dicke Version

In [97]:
alph_dicke = dicke_state(nq_qubo//2, na_target).to_gate()
beta_dicke = dicke_state(nq_qubo//2, nb_target).to_gate()

full_dicke = QuantumCircuit(nq_qubo)
full_dicke.append(alph_dicke, [2*qi for qi in range(nq_qubo//2)])
full_dicke.append(beta_dicke, [2*qi+1 for qi in range(nq_qubo//2)])

full_dicke = transpile(full_dicke, basis_gates=['id', 'ry', 'rx', 'cx'], optimization_level=1, seed_transpiler=1) 

In [98]:
# Show output of Dicke circuit 
simulator_aerlegacy = StatevectorSimulator()
result_aerlegacy = simulator_aerlegacy.run(full_dicke.decompose().decompose())

statevec_dicke = result_aerlegacy.result().get_statevector().data.reshape([-1, 1])
statevec_dicke[np.abs(statevec_dicke)<1e-15] = 0 # zero out vanishingly small terms!
psi_dicke = QuantumState.from_array(statevec_dicke)
psi_dicke

 0.333+0.000j |001111> +
 0.333+0.000j |011011> +
 0.333+0.000j |011110> +
 0.333+0.000j |100111> +
 0.333+0.000j |101101> +
 0.333+0.000j |110011> +
 0.333+0.000j |110110> +
 0.333+0.000j |111001> +
 0.333+0.000j |111100>

In [99]:
## base object for Dicke GAS
gas_obj_dicke = GAS_dicke( nq_qubo, 
                        nq_fx,
                        linear_rounded,
                        quad_rounded,
                        full_dicke)

In [178]:
def print_state(state):
    latex_string = ""
    for state in state:
        bitstring = ''.join(list(map(str, state.state_matrix[0].tolist())))

        latex_string+="$"
        if bitstring == "111100":
            latex_string += " \\mathbf{"

        latex_string += f"{np.round(state.state_op.coeff_vec[0].real, 3)} "
        latex_string += "\\ket{"
        latex_string += bitstring

        if bitstring == "111100":
            latex_string += "}}+ $ \n"
        else:
            latex_string += "} + $ \n"

    latex_string = latex_string[:-5] + "  $"
    print(latex_string)
    return None

In [179]:
## threshold y==-4
Y=-4
print("--------------------------------")
print(f"Threshold y=={Y}")
print("--------------------------------")
for n_repeats in range(1, 5):
    full_circuit = gas_obj_dicke.build_circuit(-Y, n_repeats)


    # Simulator 
    simulator_aerlegacy = StatevectorSimulator()
    result_aerlegacy = simulator_aerlegacy.run(full_circuit.decompose().decompose())

    statevec2 = result_aerlegacy.result().get_statevector().data.reshape([-1, 1])
    statevec2[np.abs(statevec2)<1e-15] = 0 # zero out vanishingly small terms!
    psi_out_full = QuantumState.from_array(statevec2)


    psi_out_sys = QuantumState(psi_out_full.state_matrix[:, -f_op.n_qubits:][::-1],
                        np.abs(psi_out_full.state_op.coeff_vec)).cleanup().normalize.sort()


    print("grover repeats: ", n_repeats)
    print_state(psi_out_sys)
    print("--------------------------------")

--------------------------------
Threshold y==-4
--------------------------------
grover repeats:  1
$0.852 \ket{001111} + $ 
$0.185 \ket{011110} + $ 
$0.185 \ket{011011} + $ 
$0.185 \ket{100111} + $ 
$0.185 \ket{110110} + $ 
$0.185 \ket{111001} + $ 
$0.185 \ket{101101} + $ 
$0.185 \ket{110011} + $ 
$ \mathbf{0.185 \ket{111100}}  $
--------------------------------
grover repeats:  2
$0.992 \ket{001111} + $ 
$0.045 \ket{110110} + $ 
$0.045 \ket{111001} + $ 
$0.045 \ket{110011} + $ 
$0.045 \ket{101101} + $ 
$0.045 \ket{100111} + $ 
$0.045 \ket{011011} + $ 
$ \mathbf{0.045 \ket{111100}}+ $ 
$0.045 \ket{011110}   $
--------------------------------
grover repeats:  3
$0.691 \ket{001111} + $ 
$0.256 \ket{011011} + $ 
$0.256 \ket{100111} + $ 
$0.256 \ket{111001} + $ 
$0.256 \ket{110110} + $ 
$0.256 \ket{101101} + $ 
$0.256 \ket{110011} + $ 
$0.256 \ket{011110} + $ 
$ \mathbf{0.256 \ket{111100}}  $
--------------------------------
grover repeats:  4
$0.352 \ket{011110} + $ 
$0.352 \ket{011011}

In [181]:
Y=-8
print("--------------------------------")
print(f"Threshold y=={Y}")
print("--------------------------------")
for n_repeats in range(1, 5):
    full_circuit = gas_obj_dicke.build_circuit(-Y, n_repeats)


    # Simulator 
    simulator_aerlegacy = StatevectorSimulator()
    result_aerlegacy = simulator_aerlegacy.run(full_circuit.decompose().decompose())

    statevec2 = result_aerlegacy.result().get_statevector().data.reshape([-1, 1])
    statevec2[np.abs(statevec2)<1e-15] = 0 # zero out vanishingly small terms!
    psi_out_full = QuantumState.from_array(statevec2)


    psi_out_sys = QuantumState(psi_out_full.state_matrix[:, -f_op.n_qubits:][::-1],
                        np.abs(psi_out_full.state_op.coeff_vec)).cleanup().normalize.sort()


    print("grover repeats: ", n_repeats)
    print_state(psi_out_sys)
    print("--------------------------------")

--------------------------------
Threshold y==-8
--------------------------------
grover repeats:  1
$0.556 \ket{011011} + $ 
$0.556 \ket{001111} + $ 
$0.556 \ket{011110} + $ 
$ \mathbf{0.111 \ket{111100}}+ $ 
$0.111 \ket{110011} + $ 
$0.111 \ket{111001} + $ 
$0.111 \ket{101101} + $ 
$0.111 \ket{100111} + $ 
$0.111 \ket{110110}   $
--------------------------------
grover repeats:  2
$0.407 \ket{100111} + $ 
$0.407 \ket{110110} + $ 
$0.407 \ket{101101} + $ 
$0.407 \ket{111001} + $ 
$0.407 \ket{110011} + $ 
$ \mathbf{0.407 \ket{111100}}+ $ 
$0.037 \ket{011011} + $ 
$0.037 \ket{011110} + $ 
$0.037 \ket{001111}   $
--------------------------------
grover repeats:  3
$0.531 \ket{011110} + $ 
$0.531 \ket{011011} + $ 
$0.531 \ket{001111} + $ 
$0.16 \ket{100111} + $ 
$0.16 \ket{110110} + $ 
$ \mathbf{0.16 \ket{111100}}+ $ 
$0.16 \ket{111001} + $ 
$0.16 \ket{110011} + $ 
$0.16 \ket{101101}   $
--------------------------------
grover repeats:  4
$0.391 \ket{001111} + $ 
$0.391 \ket{011110} + $ 


In [182]:
Y=-12
print("--------------------------------")
print(f"Threshold y=={Y}")
print("--------------------------------")
for n_repeats in range(1, 5):
    full_circuit = gas_obj_dicke.build_circuit(-Y, n_repeats)


    # Simulator 
    simulator_aerlegacy = StatevectorSimulator()
    result_aerlegacy = simulator_aerlegacy.run(full_circuit.decompose().decompose())

    statevec2 = result_aerlegacy.result().get_statevector().data.reshape([-1, 1])
    statevec2[np.abs(statevec2)<1e-15] = 0 # zero out vanishingly small terms!
    psi_out_full = QuantumState.from_array(statevec2)


    psi_out_sys = QuantumState(psi_out_full.state_matrix[:, -f_op.n_qubits:][::-1],
                        np.abs(psi_out_full.state_op.coeff_vec)).cleanup().normalize.sort()


    print("grover repeats: ", n_repeats)
    print_state(psi_out_sys)
    print("--------------------------------")

--------------------------------
Threshold y==-12
--------------------------------
grover repeats:  1
$0.407 \ket{100111} + $ 
$0.407 \ket{011011} + $ 
$0.407 \ket{011110} + $ 
$0.407 \ket{001111} + $ 
$ \mathbf{0.259 \ket{111100}}+ $ 
$0.259 \ket{110011} + $ 
$0.259 \ket{111001} + $ 
$0.259 \ket{110110} + $ 
$0.259 \ket{101101}   $
--------------------------------
grover repeats:  2
$0.391 \ket{110110} + $ 
$0.391 \ket{101101} + $ 
$0.391 \ket{111001} + $ 
$0.391 \ket{110011} + $ 
$ \mathbf{0.391 \ket{111100}}+ $ 
$0.243 \ket{001111} + $ 
$0.243 \ket{011011} + $ 
$0.243 \ket{011110} + $ 
$0.243 \ket{100111}   $
--------------------------------
grover repeats:  3
$0.461 \ket{001111} + $ 
$0.461 \ket{011011} + $ 
$0.461 \ket{011110} + $ 
$0.461 \ket{100111} + $ 
$0.172 \ket{110110} + $ 
$0.172 \ket{111001} + $ 
$0.172 \ket{101101} + $ 
$0.172 \ket{110011} + $ 
$ \mathbf{0.172 \ket{111100}}  $
--------------------------------
grover repeats:  4
$0.429 \ket{101101} + $ 
$0.429 \ket{110011

In [183]:
Y=-16
print("--------------------------------")
print(f"Threshold y=={Y}")
print("--------------------------------")
for n_repeats in range(1, 5):
    full_circuit = gas_obj_dicke.build_circuit(-Y, n_repeats)


    # Simulator 
    simulator_aerlegacy = StatevectorSimulator()
    result_aerlegacy = simulator_aerlegacy.run(full_circuit.decompose().decompose())

    statevec2 = result_aerlegacy.result().get_statevector().data.reshape([-1, 1])
    statevec2[np.abs(statevec2)<1e-15] = 0 # zero out vanishingly small terms!
    psi_out_full = QuantumState.from_array(statevec2)


    psi_out_sys = QuantumState(psi_out_full.state_matrix[:, -f_op.n_qubits:][::-1],
                        np.abs(psi_out_full.state_op.coeff_vec)).cleanup().normalize.sort()


    print("grover repeats: ", n_repeats)
    print_state(psi_out_sys)
    print("--------------------------------")

--------------------------------
Threshold y==-16
--------------------------------
grover repeats:  1
$0.556 \ket{111001} + $ 
$0.556 \ket{110011} + $ 
$ \mathbf{0.556 \ket{111100}}+ $ 
$0.111 \ket{110110} + $ 
$0.111 \ket{001111} + $ 
$0.111 \ket{100111} + $ 
$0.111 \ket{101101} + $ 
$0.111 \ket{011011} + $ 
$0.111 \ket{011110}   $
--------------------------------
grover repeats:  2
$0.407 \ket{011110} + $ 
$0.407 \ket{011011} + $ 
$0.407 \ket{100111} + $ 
$0.407 \ket{001111} + $ 
$0.407 \ket{101101} + $ 
$0.407 \ket{110110} + $ 
$ \mathbf{0.037 \ket{111100}}+ $ 
$0.037 \ket{111001} + $ 
$0.037 \ket{110011}   $
--------------------------------
grover repeats:  3
$0.531 \ket{110011} + $ 
$0.531 \ket{111001} + $ 
$ \mathbf{0.531 \ket{111100}}+ $ 
$0.16 \ket{001111} + $ 
$0.16 \ket{101101} + $ 
$0.16 \ket{110110} + $ 
$0.16 \ket{011011} + $ 
$0.16 \ket{011110} + $ 
$0.16 \ket{100111}   $
--------------------------------
grover repeats:  4
$0.391 \ket{110011} + $ 
$0.391 \ket{111001} + $ 

In [185]:
Y=-20
print("--------------------------------")
print(f"Threshold y=={Y}")
print("--------------------------------")
for n_repeats in range(1, 5):
    full_circuit = gas_obj_dicke.build_circuit(-Y, n_repeats)


    # Simulator 
    simulator_aerlegacy = StatevectorSimulator()
    result_aerlegacy = simulator_aerlegacy.run(full_circuit.decompose().decompose())

    statevec2 = result_aerlegacy.result().get_statevector().data.reshape([-1, 1])
    statevec2[np.abs(statevec2)<1e-15] = 0 # zero out vanishingly small terms!
    psi_out_full = QuantumState.from_array(statevec2)


    psi_out_sys = QuantumState(psi_out_full.state_matrix[:, -f_op.n_qubits:][::-1],
                        np.abs(psi_out_full.state_op.coeff_vec)).cleanup().normalize.sort()


    print("grover repeats: ", n_repeats)
    print_state(psi_out_sys)
    print("--------------------------------")

--------------------------------
Threshold y==-20
--------------------------------
grover repeats:  1
$ \mathbf{0.852 \ket{111100}}+ $ 
$0.185 \ket{001111} + $ 
$0.185 \ket{100111} + $ 
$0.185 \ket{011011} + $ 
$0.185 \ket{011110} + $ 
$0.185 \ket{101101} + $ 
$0.185 \ket{110011} + $ 
$0.185 \ket{111001} + $ 
$0.185 \ket{110110}   $
--------------------------------
grover repeats:  2
$ \mathbf{0.992 \ket{111100}}+ $ 
$0.045 \ket{100111} + $ 
$0.045 \ket{001111} + $ 
$0.045 \ket{011011} + $ 
$0.045 \ket{111001} + $ 
$0.045 \ket{011110} + $ 
$0.045 \ket{110110} + $ 
$0.045 \ket{101101} + $ 
$0.045 \ket{110011}   $
--------------------------------
grover repeats:  3
$ \mathbf{0.691 \ket{111100}}+ $ 
$0.256 \ket{011011} + $ 
$0.256 \ket{011110} + $ 
$0.256 \ket{001111} + $ 
$0.256 \ket{100111} + $ 
$0.256 \ket{110110} + $ 
$0.256 \ket{110011} + $ 
$0.256 \ket{111001} + $ 
$0.256 \ket{101101}   $
--------------------------------
grover repeats:  4
$0.352 \ket{001111} + $ 
$0.352 \ket{100111